# 04 — RQ3 Cross-Generation Evaluation: Variant C (HINTED) on GPT-3.5-turbo

**Purpose.** Evaluate whether the prompt-engineering gains observed for GPT-4o-mini in `02_run_ablation_hinted.ipynb` generalize across LLM generations. We apply the highest-performing HINTED variant — Variant C (Patterns) — on a stratified 100-sample subset using GPT-3.5-turbo. The contrast between GPT-4o-mini's F1 = 0.931 and GPT-3.5-turbo's near-uniform Safe predictions (Recall ≈ 0.022) is the empirical basis for the Cognitive Threshold finding (Section 4.3 of the paper).

**Sample.** 100 samples drawn from the 234-sample benchmark, stratified by category: 50 CWE-89 (SQLi), 42 CWE-78 (CmdInj), 8 Safe. The exact 100 files used in the paper are listed in `data/file_list_100_stratified.txt`, which is committed to the repository to ensure reproducibility.

**Inputs.**
- `<BASE_DIR>/final_dataset/` — produced by `01_dataset_curation.ipynb`.
- `data/file_list_100_stratified.txt` — committed to the repository.
- `prompts/variant_C_patterns_hinted.txt` — same prompt used in `02_run_ablation_hinted.ipynb`.
- `prompts/system_message.txt` — committed to the repository.
- An OpenAI API key with access to `gpt-3.5-turbo`.

**Outputs.**
- `<BASE_DIR>/results/rq3_cross_generation_gpt35.csv` — one row per sample, with the model's prediction.

**Cost & runtime.** 100 samples × 1 variant on `gpt-3.5-turbo`. Approximate cost: USD 0.02–0.05. Approximate runtime: 5–10 minutes. Resume-on-interrupt is supported.

**Note on terminology.** This notebook is named `04_run_cross_generation` but the produced CSV is named `rq3_cross_generation_gpt35.csv` to align with the paper's RQ3. Earlier exploratory runs in the source codebase used the label `rq2_generalization`; the canonical name in this repository follows the paper.


## 1. Setup

Same setup as `02` and `03`.

In [ ]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ---- USER-EDITABLE ----
BASE_DIR_OVERRIDE = None  # e.g., Path('/content/drive/MyDrive/llm-vuln-detection-ablation')
MODEL_ID          = 'gpt-3.5-turbo'
TEMPERATURE       = 0.1
MAX_RETRIES       = 3
RETRY_BACKOFF_SEC = 2
# -----------------------

def find_repo_root() -> Path:
    """Locate the repository root by walking upward from the current working directory."""
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT         = find_repo_root()
BASE_DIR          = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')
DATASET_DIR       = BASE_DIR / 'final_dataset'
RESULTS_DIR       = BASE_DIR / 'results'
PROMPTS_DIR       = REPO_ROOT / 'prompts'
DATA_DIR          = REPO_ROOT / 'data'
OUTPUT_CSV        = RESULTS_DIR / 'rq3_cross_generation_gpt35.csv'
STRATIFIED_LIST   = DATA_DIR / 'file_list_100_stratified.txt'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'REPO_ROOT       : {REPO_ROOT}')
print(f'DATASET_DIR     : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'PROMPTS_DIR     : {PROMPTS_DIR}  (exists: {PROMPTS_DIR.exists()})')
print(f'STRATIFIED_LIST : {STRATIFIED_LIST}  (exists: {STRATIFIED_LIST.exists()})')
print(f'OUTPUT_CSV      : {OUTPUT_CSV}')
print(f'MODEL_ID        : {MODEL_ID}')


## 2. Load prompt and the 100-sample subset

Read the Variant C HINTED prompt (the same one used in `02_run_ablation_hinted.ipynb`) and the canonical list of 100 stratified samples. The list is committed to the repository to guarantee that re-runs evaluate exactly the same samples as the paper.

In [ ]:
def load_prompt(filename: str) -> str:
    """Read a prompt file and strip a single trailing newline."""
    with open(PROMPTS_DIR / filename, encoding='utf-8') as f:
        return f.read().rstrip('\n')

PROMPT_NAME    = 'Variant_C_Patterns'
PROMPT_TEXT    = load_prompt('variant_C_patterns_hinted.txt')
SYSTEM_MESSAGE = load_prompt('system_message.txt')

print(f'{PROMPT_NAME:25s} ({len(PROMPT_TEXT):3d} chars): {PROMPT_TEXT[:80]}...')
print(f'{"SYSTEM_MESSAGE":25s} ({len(SYSTEM_MESSAGE):3d} chars): {SYSTEM_MESSAGE[:80]}...')
print()

# Read the stratified subset (relative paths, e.g., 'CWE_89_SQLi/<filename>.php').
with open(STRATIFIED_LIST) as f:
    subset_relpaths = [line.strip() for line in f if line.strip()]
print(f'Stratified subset size: {len(subset_relpaths)}')

# Resolve to absolute paths under DATASET_DIR.
subset_paths = [DATASET_DIR / rp for rp in subset_relpaths]
missing = [p for p in subset_paths if not p.exists()]
if missing:
    raise RuntimeError(
        f'{len(missing)} samples listed in {STRATIFIED_LIST.name} are not present under {DATASET_DIR}. '
        f'Run 01_dataset_curation.ipynb first. Example missing: {missing[0]}'
    )
print(f'All {len(subset_paths)} samples located under {DATASET_DIR}')


## 3. Run the cross-generation evaluation

Query GPT-3.5-turbo once per sample with Variant C (HINTED). The output CSV is written incrementally and supports resume-on-interrupt.

In [ ]:
def get_true_label(filepath: Path) -> tuple:
    """Infer (true_label, true_cwe) from the parent directory name."""
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '89' in folder:
        return 'Vulnerable', 'CWE-89'
    if '78' in folder:
        return 'Vulnerable', 'CWE-78'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

def predict(prompt_text: str, code_content: str) -> str:
    """Send one (prompt, code) pair to the model with retry. Return 'Vulnerable' / 'Safe' / 'Error'."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                response_format={'type': 'json_object'},
                temperature=TEMPERATURE,
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
                ],
            )
            result = json.loads(response.choices[0].message.content)
            return result.get('prediction', 'Error')
        except Exception:
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming from existing CSV: {len(processed)} samples already processed.')
else:
    columns = ['File_Name', 'True_Label', 'Prediction', 'True_CWE']
    results_df = pd.DataFrame(columns=columns)
    processed = set()

to_process = [p for p in subset_paths if p.name not in processed]
print(f'Samples remaining to process: {len(to_process)}')

for filepath in tqdm(to_process, desc=f'Cross-generation on {MODEL_ID}'):
    true_label, true_cwe = get_true_label(filepath)
    code_content = filepath.read_text(encoding='utf-8', errors='ignore')

    prediction = predict(PROMPT_TEXT, code_content)
    row = {
        'File_Name': filepath.name,
        'True_Label': true_label,
        'Prediction': prediction,
        'True_CWE': true_cwe,
    }

    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nCross-generation evaluation complete. Results written to {OUTPUT_CSV}')


## 4. Inspect results

Quick sanity checks on the output CSV. The headline finding from the paper — that GPT-3.5-turbo classifies the vast majority of samples as `Safe` regardless of true label — should be visible in the prediction distribution. Detailed metric computation is deferred to `05_metrics_and_figures.ipynb`.

In [ ]:
df = pd.read_csv(OUTPUT_CSV)

print(f'Total rows           : {len(df)}')
print(f'Expected             : 100')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
print('True CWE distribution:')
print(df['True_CWE'].value_counts(dropna=False).to_string())
print()
print('Prediction distribution:')
print(df['Prediction'].value_counts().to_string())
print()
# Quick crosstab
print('Cross-tabulation (rows = True_Label, cols = Prediction):')
print(pd.crosstab(df['True_Label'], df['Prediction']).to_string())
